In [1]:
# imports

import pandas as pd
import numpy as np
import json
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import polars as pl
from pathlib import Path
from sklearn.metrics import accuracy_score


In [2]:
# Helpers to read in data and convert to dataframe

def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def convert_to_dataframe(data, is_full_cleaned=False):
    record = []
    for table_id, details in data.items():
        statements = details[0]
        enatiled_statuses = details[1]
        caption = details[-1]
        
        if is_full_cleaned:
            pos_tags = details[2]
            for statement, entailed_status, statement_pos_tags in zip(statements, enatiled_statuses, pos_tags):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "entailed_status": entailed_status,
                    "caption": caption,
                    "pos_tags": statement_pos_tags
                }) 
        else:
            for statement, entailed_status in zip(statements, enatiled_statuses):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "entailed_status": entailed_status,
                    "caption": caption
                })
    return pd.DataFrame(record)

def load_and_convert_data_to_dataframe(path, is_full_cleaned=False):
    raw_data = load_json_data(path)
    
    return convert_to_dataframe(raw_data, is_full_cleaned=is_full_cleaned)

In [3]:
# Get table dataframes
df_tables = (
    pl.read_csv(
        "data/tables/*.csv",
        has_header=False,
        separator="\1", # Non-existent delimiter so full line/file stays intact
        new_columns=["table_content"],
        include_file_paths="file_name",
        truncate_ragged_lines=True 
    )
    .group_by("file_name", maintain_order=True) # Group all the table lines together 
    .agg(pl.col("table_content").str.join("\n"))
    .to_pandas()
)

df_tables["file_name"] = df_tables["file_name"].apply(lambda p: Path(p).name) # only keep the table id

df_tables.shape

(16573, 2)

In [4]:
df_tables.iloc[0, 1]

'aircraft#description#max gross weight#total disk area#max disk loading\nrobinson r - 22#light utility helicopter#1370 lb (635 kg)#497 ft square (46.2 m square)#2.6 lb / ft square (14 kg / m square)\nbell 206b3 jetranger#turboshaft utility helicopter#3200 lb (1451 kg)#872 ft square (81.1 m square)#3.7 lb / ft square (18 kg / m square)\nch - 47d chinook#tandem rotor helicopter#50000 lb (22680 kg)#5655 ft square (526 m square)#8.8 lb / ft square (43 kg / m square)\nmil mi - 26#heavy - lift helicopter#123500 lb (56000 kg)#8495 ft square (789 m square)#14.5 lb / ft square (71 kg / m square)\nch - 53e super stallion#heavy - lift helicopter#73500 lb (33300 kg)#4900 ft square (460 m square)#15 lb / ft square (72 kg / m square)'

In [5]:
# Load in data frames

df_full_cleaned = load_and_convert_data_to_dataframe("data/full_cleaned.json", is_full_cleaned=True)

df_full_cleaned_refuted = df_full_cleaned[df_full_cleaned['entailed_status'] == 0]
df_full_cleaned_entailed = df_full_cleaned[df_full_cleaned['entailed_status'] == 1]

df_r1_training = load_and_convert_data_to_dataframe("data/r1_training_all.json")

df_r2_training = load_and_convert_data_to_dataframe("data/r2_training_all.json")

df_training = load_and_convert_data_to_dataframe("data/train_examples.json")

df_test = load_and_convert_data_to_dataframe("data/test_examples.json")

df_validation = load_and_convert_data_to_dataframe("data/val_examples.json")


In [12]:
# returns a pandas dataframe that contains a dataframe linking the table content with a statement and entailed status
def get_table_claim_df(df, whole_table=True, table_before_claim=True):
    if table_before_claim:
        df_table_claim = pd.merge(
            df_tables[["file_name", "table_content"]],
            df[["table_id", "statement", "entailed_status"]],
            left_on="file_name",
            right_on="table_id",
            how="inner" # join matching pairs
        )
    else:
        df_table_claim = pd.merge(
            df[["table_id", "statement", "entailed_status"]],
            df_tables[["file_name", "table_content"]],
            left_on="table_id",
            right_on="file_name",
            how="inner" # join matching pairs
        )

    if table_before_claim:
        df_table_claim["X_val"] = "TABLE: " + df_table_claim["table_content"] + " | CLAIM: " + df_table_claim["statement"]
    else:
        df_table_claim["X_val"] = "CLAIM: " + df_table_claim["statement"] + " | TABLE: " + df_table_claim["table_content"]

    return df_table_claim

# Get training table-claim corpus for tfidf vectorizer
df_train_table_claim = get_table_claim_df(df_training,)
train_table_claim_corpus = df_train_table_claim["X_val"].to_numpy()

# Get training table-claim embeddings and truth labels
vectorizer = TfidfVectorizer(max_features=30000)
X_train = vectorizer.fit_transform(train_table_claim_corpus)
y_train = df_train_table_claim["entailed_status"]

# Get test table-claim corpus for tfidf vectorizer
df_test_table_claim = get_table_claim_df(df_test)
test_table_claim_corpus = df_test_table_claim["X_val"].to_numpy()

# Get test table-claim embeddings and truth labels
X_test = vectorizer.transform(test_table_claim_corpus)
y_test = df_test_table_claim["entailed_status"]

print(X_test.shape)
print(y_test.shape)

(12779, 30000)
(12779,)


In [13]:
logistic_regression_model = LogisticRegression(solver="lbfgs", max_iter=10000, random_state=0).fit(X_train, y_train)
y_test_pred = logistic_regression_model.predict(X_test)
accuracy_score(y_test, y_test_pred)

0.524062915721105